# 04 SVM Aspect Classification

Finalization notebook for Phase 9C. This notebook documents the SVM aspect-classifier decision and candidate AHP/Fuzzy AHP criteria. It is read-only: it does not retrain SVM, fit TF-IDF, or create model artifacts.


## CRISP-DM Stage

Model evaluation and transition to decision-support design. The selected SVM scenario is finalized so the project can proceed to AHP/Fuzzy AHP criteria and expert judgement design.


## SVM Role in SentiRank

SVM is used for aspect classification, not sentiment classification. IndoBERT `run_3_weighted_loss_lr_1e-5` remains the final sentiment classifier candidate. The SVM aspect classifier converts review text into actionable issue categories that can support priority-ranking criteria after expert validation.


## Weak Label Limitation

The SVM aspect classifier was trained and evaluated on weak labels derived from keyword-based aspect labeling. Therefore, the evaluation reflects the model's ability to learn weak-label patterns, not expert-validated ground truth.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "ml-service").exists():
            return candidate
    raise RuntimeError("Could not find SentiRank project root from current working directory.")


PROJECT_ROOT = find_project_root()
EDA04_DIR = PROJECT_ROOT / "datasets" / "outputs" / "eda" / "04_svm"
FIG04_DIR = PROJECT_ROOT / "docs" / "figures" / "04_svm"
MODEL_DIR = PROJECT_ROOT / "ml-service" / "saved_models" / "svm"
print(f"Project root: {PROJECT_ROOT}")


## Load Final Decision Outputs

The final decision files are generated from existing Phase 9B metrics. Missing files are reported clearly instead of causing silent failures.


In [ ]:
def load_json(path: Path):
    if not path.exists():
        display(Markdown(f"Missing JSON: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path):
    if not path.exists():
        display(Markdown(f"Missing CSV: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    return pd.read_csv(path)


final_selection = load_json(EDA04_DIR / "svm_final_model_selection.json")
selection_table = load_csv(EDA04_DIR / "svm_final_model_selection.csv")
taxonomy = load_json(EDA04_DIR / "final_aspect_taxonomy_for_ahp.json")
taxonomy_table = load_csv(EDA04_DIR / "final_aspect_taxonomy_for_ahp.csv")
artifact_manifest = load_json(EDA04_DIR / "svm_artifact_manifest.json")

if final_selection:
    display(Markdown(f"Selected SVM scenario: `{final_selection['selected_scenario']}`"))
    display(pd.json_normalize(final_selection["selected_metrics"]).T.reset_index().rename(columns={"index": "metric", 0: "value"}))
if selection_table is not None:
    display(selection_table)


## Scenario Comparison

`original_7class` is retained as an exploratory baseline. `merged_5class` is selected because it improves overall performance and reduces minority-class instability.


In [ ]:
comparison = load_csv(EDA04_DIR / "svm_scenario_comparison.csv")
if comparison is not None:
    display(comparison)

if final_selection:
    display(Markdown("### Selection reasons"))
    display(Markdown("\n".join(f"- {reason}" for reason in final_selection["selection_reason"])))
    display(Markdown(f"### Limitation\n{final_selection['limitations']}"))


## Classification Reports and Confusion Matrices

These tables preserve the detailed Phase 9B evaluation evidence for both scenarios.


In [ ]:
for scenario in ["original_7class", "merged_5class"]:
    report = load_json(EDA04_DIR / f"svm_{scenario}_classification_report.json")
    confusion = load_csv(EDA04_DIR / f"svm_{scenario}_confusion_matrix.csv")
    if report:
        rows = []
        for label, values in report.items():
            if isinstance(values, dict):
                rows.append({"label": label, **values})
        display(Markdown(f"### {scenario} classification report"))
        display(pd.DataFrame(rows))
    if confusion is not None:
        display(Markdown(f"### {scenario} confusion matrix"))
        display(confusion)


## Final 5 Aspect Taxonomy for AHP/Fuzzy AHP

The 5-class taxonomy is a candidate criteria set only. Expert judgement is still required before final AHP/Fuzzy AHP weighting.


In [ ]:
if taxonomy:
    display(Markdown(taxonomy["note"]))
if taxonomy_table is not None:
    display(taxonomy_table)


## Why UI/UX and Audio Quality Were Merged

`Audio Quality` had limited standalone support and conceptually belongs with content/listening experience, so it is merged with `Features & Content`. `UI/UX` was also sparse and closely related to interaction reliability, so it is merged with `Performance & Stability` into `App Reliability & Usability`. These merges improve minority-class stability and reduce AHP/Fuzzy AHP pairwise-comparison burden.


## Figures

The figures below are existing Phase 9B evaluation artifacts. This notebook only displays them.


In [ ]:
def show_figures(paths: list[Path]) -> None:
    for path in paths:
        if path.exists():
            display(Markdown(f"**{path.relative_to(PROJECT_ROOT)}**"))
            display(Image(filename=str(path)))
        else:
            display(Markdown(f"Missing figure: `{path.relative_to(PROJECT_ROOT)}`"))


show_figures([
    FIG04_DIR / "svm_original_7class_confusion_matrix.png",
    FIG04_DIR / "svm_original_7class_class_f1.png",
    FIG04_DIR / "svm_merged_5class_confusion_matrix.png",
    FIG04_DIR / "svm_merged_5class_class_f1.png",
    FIG04_DIR / "svm_scenario_comparison.png",
])


## Artifact Manifest

SVM model artifacts are local generated files and must remain ignored by Git.


In [ ]:
if artifact_manifest:
    display(pd.DataFrame(artifact_manifest["expected_artifacts"]))
    display(Markdown(f"Selected pipeline: `{artifact_manifest['selected_pipeline']}`"))
    display(Markdown(f"Selected label mapping: `{artifact_manifest['selected_label_mapping']}`"))
    display(Markdown(f"Artifact Git policy: `{artifact_manifest['artifact_git_policy']}`"))


## Next Step

Proceed to Phase 10A: AHP/Fuzzy AHP Criteria and Expert Judgement Design. The criteria should be validated by experts before final pairwise weighting and ranking comparison are implemented.
